# THINGS ResNet-50 Full Training

Use this notebook only after the smoke run has passed and a GPU runtime is attached. It runs the real image-only baseline training, copies checkpoints/logs to Drive, then can extract and evaluate embeddings.

This notebook intentionally has no smoke-training cell. If CUDA is not available, stop immediately.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/sabszh/human-things.git"
PROJECT_ROOT = Path("/content/human-things")

RESET_LOCAL_REPO = True
USE_DRIVE_DATA_ZIP = True
DRIVE_DATA_ZIP = Path("/content/drive/MyDrive/human-things-data/human-things-data.zip")
DRIVE_DATA_FILE_ID = "1OofSEPS34SA6Jol3OIqO208ekIHz1UEF"
LOCAL_DATA_ZIP = Path("/content/human-things-data.zip")

HEAD_EPOCHS = 5
LAYER4_EPOCHS = 10
BATCH_SIZE = 64
NUM_WORKERS = 2
PROGRESS_EVERY_BATCHES = 25

RUN_TRAINING = True
RUN_EXTRACT_AND_EVAL = True
COPY_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/human-things-runs/full_baseline_resnet50")

print("Full-training configuration", flush=True)
print("- repo:", REPO_URL, flush=True)
print("- project root:", PROJECT_ROOT, flush=True)
print("- Drive zip:", DRIVE_DATA_ZIP, flush=True)
print("- Drive file id:", DRIVE_DATA_FILE_ID, flush=True)
print("- epochs:", {"head": HEAD_EPOCHS, "layer4": LAYER4_EPOCHS}, flush=True)
print("- batch size:", BATCH_SIZE, flush=True)
print("- num workers:", NUM_WORKERS, flush=True)
print("- progress every batches:", PROGRESS_EVERY_BATCHES, flush=True)
print("- run training:", RUN_TRAINING, flush=True)
print("- run extract/eval:", RUN_EXTRACT_AND_EVAL, flush=True)
print("- Drive output root:", DRIVE_OUTPUT_ROOT, flush=True)


## 1. GPU Runtime Check

This must print `CUDA: True` and `Tesla T4` or another CUDA GPU. If it prints CPU, stop and change runtime type.


In [ ]:
import torch
print("Python:", sys.version, flush=True)
print("Torch:", torch.__version__, flush=True)
print("CUDA:", torch.cuda.is_available(), flush=True)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", flush=True)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Stop here and switch Colab to a GPU runtime.")


## 2. Mount Drive


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped or failed:", repr(exc), flush=True)


## 3. Clone Repo


In [ ]:
def run_checked(command, cwd=None, check=True):
    print("\n$", " ".join(map(str, command)), flush=True)
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.stderr:
        print(result.stderr, flush=True)
    print("exit code:", result.returncode, flush=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, command))}")
    return result

print("Step 3: preparing repo checkout", flush=True)
os.chdir("/content")
remote = run_checked(["git", "ls-remote", "--heads", REPO_URL], check=False)
if remote.returncode != 0:
    raise RuntimeError("Cannot access GitHub repo. Read git output above.")

if RESET_LOCAL_REPO:
    print(f"Removing temporary checkout: {PROJECT_ROOT}", flush=True)
    shutil.rmtree(PROJECT_ROOT, ignore_errors=True)

if not PROJECT_ROOT.exists():
    run_checked(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])
else:
    pull = run_checked(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=False)
    if pull.returncode != 0:
        shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
        run_checked(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])

os.chdir(PROJECT_ROOT)
print("Project root:", Path.cwd(), flush=True)
run_checked(["git", "rev-parse", "--short", "HEAD"])
run_checked(["find", "scripts", "-maxdepth", "1", "-type", "f", "-print"])


## 4. Install Requirements


In [ ]:
print("Step 4: installing requirements", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Requirements installed.", flush=True)

import torch
print("Torch after install:", torch.__version__, flush=True)
print("CUDA after install:", torch.cuda.is_available(), flush=True)
print("Device after install:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", flush=True)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA disappeared after installing requirements. Stop; do not train on CPU.")


## 5. Restore Data

Uses the Drive zip if present, otherwise downloads the shared Drive file by ID. This avoids OSF during full training.


In [ ]:
def restore_zip(zip_path):
    print(f"Restoring data from zip: {zip_path}", flush=True)
    print(f"Zip size GB: {zip_path.stat().st_size / 1e9:.2f}", flush=True)
    data_dir = PROJECT_ROOT / "data"
    if data_dir.exists():
        print(f"Removing existing data folder: {data_dir}", flush=True)
        shutil.rmtree(data_dir)
    print("Unzipping. This can take several minutes for ~28k image files.", flush=True)
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.namelist()
        print(f"Zip entries: {len(members)}", flush=True)
        archive.extractall(PROJECT_ROOT)
    print("Restored:", data_dir, flush=True)

print("Step 5: restoring data", flush=True)
zip_to_restore = None
if USE_DRIVE_DATA_ZIP and DRIVE_DATA_ZIP.exists():
    print("Found mounted Drive zip.", flush=True)
    zip_to_restore = DRIVE_DATA_ZIP
elif LOCAL_DATA_ZIP.exists():
    print("Found local Colab zip.", flush=True)
    zip_to_restore = LOCAL_DATA_ZIP
elif DRIVE_DATA_FILE_ID:
    print(f"Downloading shared Drive file to: {LOCAL_DATA_ZIP}", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    subprocess.run([sys.executable, "-m", "gdown", "--id", DRIVE_DATA_FILE_ID, "-O", str(LOCAL_DATA_ZIP)], check=True)
    zip_to_restore = LOCAL_DATA_ZIP
else:
    raise FileNotFoundError("No Drive zip or file ID configured.")

restore_zip(zip_to_restore)
print("Data restore finished.", flush=True)


## 6. Verify Data and Splits


In [ ]:
import pandas as pd

print("Step 6: verifying data", flush=True)
concepts = pd.read_csv(PROJECT_ROOT / "data/processed/concepts.csv")
images = pd.read_csv(PROJECT_ROOT / "data/processed/images.csv")
things_dir = PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGS/object_images"
cc0_dir = PROJECT_ROOT / "data/raw/THINGS-database/osfstorage/images_THINGSplus-CC0/object_images_CC0"
things_count = sum(1 for _ in things_dir.rglob("*.jpg")) if things_dir.exists() else 0
cc0_count = sum(1 for _ in cc0_dir.rglob("*.jpg")) if cc0_dir.exists() else 0
print("concepts rows:", len(concepts), flush=True)
print("concept_index unique:", concepts["concept_index"].nunique(), flush=True)
print("images rows:", len(images), flush=True)
print("image concept_index nulls:", int(images["concept_index"].isna().sum()), flush=True)
print("image concept_index unique:", images["concept_index"].nunique(), flush=True)
print("THINGS jpg count:", things_count, flush=True)
print("THINGSplus CC0 jpg count:", cc0_count, flush=True)
assert len(concepts) == 1854
assert concepts["concept_index"].nunique() == 1854
assert images["concept_index"].isna().sum() == 0
assert images["concept_index"].nunique() == 1854
assert things_count == 26107

print("Rebuilding metadata and splits", flush=True)
subprocess.run([sys.executable, "scripts/01_make_metadata_csv.py"], check=True)
subprocess.run([sys.executable, "scripts/02_make_image_splits.py"], check=True)
splits = pd.read_csv(PROJECT_ROOT / "data/baseline/image_splits.csv")
print("split counts:", splits["split"].value_counts().to_dict(), flush=True)
print("concepts per split:", splits.groupby("split")["concept_id"].nunique().to_dict(), flush=True)
print("missing image files:", int((~splits["image_exists"].astype(bool)).sum()), flush=True)
assert splits["concept_id"].nunique() == 1854
assert int((~splits["image_exists"].astype(bool)).sum()) == 0
print("Data and split verification passed.", flush=True)


## 7. Dry-Run Loader


In [ ]:
print("Step 7: dataloader dry-run", flush=True)
subprocess.run([
    sys.executable,
    "scripts/03_train_resnet50_image_only.py",
    "--dry-run",
    "--batch-size", "64",
    "--num-workers", str(NUM_WORKERS),
], check=True)
print("Dataloader dry-run passed.", flush=True)


## 8. Full Training

This is the real run. It does not use `--max-train-batches` or `--max-eval-batches`. Progress prints every `PROGRESS_EVERY_BATCHES` batches.


In [ ]:
if RUN_TRAINING:
    print("Step 8: running full training", flush=True)
    print("Expected: this may take hours on a T4.", flush=True)
    cmd = [
        sys.executable,
        "scripts/03_train_resnet50_image_only.py",
        "--head-epochs", str(HEAD_EPOCHS),
        "--layer4-epochs", str(LAYER4_EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--progress-every-batches", str(PROGRESS_EVERY_BATCHES),
    ]
    print("Running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    print("Full training finished.", flush=True)
else:
    print("Skipping full training.", flush=True)


## 9. Inspect and Copy Training Outputs


In [ ]:
out = PROJECT_ROOT / "outputs/baseline_resnet50"
print("Step 9: inspecting training outputs", flush=True)
for path in [out / "training_log.csv", out / "metrics.json", out / "best_model.pt", out / "last_model.pt"]:
    if path.exists():
        print(f"exists: {path} size={path.stat().st_size}", flush=True)
    else:
        print(f"missing: {path}", flush=True)

log = out / "training_log.csv"
if log.exists():
    print("Last training log lines:", flush=True)
    print("\n".join(log.read_text().splitlines()[-20:]), flush=True)
metrics = out / "metrics.json"
if metrics.exists():
    print("Metrics:", flush=True)
    print(metrics.read_text()[:4000], flush=True)

if COPY_OUTPUTS_TO_DRIVE and Path("/content/drive/MyDrive").exists():
    print("Copying training outputs to Drive", flush=True)
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for name in ["outputs/baseline_resnet50", "data/baseline", "data/processed"]:
        src = PROJECT_ROOT / name
        dst = DRIVE_OUTPUT_ROOT / name
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"Copied {src} -> {dst}", flush=True)
    print("Drive copy finished:", DRIVE_OUTPUT_ROOT, flush=True)
else:
    print("Skipping Drive copy.", flush=True)


## 10. Extract and Evaluate Embeddings

Run this after full training has finished and checkpoints are copied.


In [ ]:
if RUN_EXTRACT_AND_EVAL:
    print("Step 10: extracting embeddings and evaluating", flush=True)
    subprocess.run([
        sys.executable,
        "scripts/04_extract_resnet50_embeddings.py",
        "--batch-size", "128",
        "--num-workers", str(NUM_WORKERS),
    ], check=True)
    subprocess.run([sys.executable, "scripts/05_evaluate_resnet50_embeddings.py"], check=True)
    print("Extraction/evaluation finished.", flush=True)

    if COPY_OUTPUTS_TO_DRIVE and Path("/content/drive/MyDrive").exists():
        DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        src = PROJECT_ROOT / "outputs/baseline_resnet50"
        dst = DRIVE_OUTPUT_ROOT / "outputs/baseline_resnet50"
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Copied final outputs {src} -> {dst}", flush=True)
else:
    print("Skipping extraction/evaluation.", flush=True)
